<a href="https://colab.research.google.com/github/Lupo6789/tesi_scacchi/blob/main/02_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In questa prima fase del Feature Engineering, vengono caricati separatamente i dataset di addestramento e valutazione, mantenendo la rigida separazione stabilita nel Notebook 01.
Viene calcolata anche un'altra statistica, "numero di mosse reali" che verrà cancellata prima della generazione finale dei file, serve solo come statistica ausiliare per il codice dopo. Viene creata la feature relativa al tempo stimato della partita

In [ ]:
import pandas as pd
import numpy as np

print("1. CARICAMENTO E PREPARAZIONE DATI")

# 1. Definizione della formula ufficiale Lichess
# Il formato di 'increment_code' è "Minuti+Secondi" (es. "10+5")
# Formula Lichess: (MinutiBase * 60) + (Incremento * 40)
def stima_tempo_lichess(increment_code):
    try:
        base_min, inc_sec = increment_code.split('+')
        tempo_totale_sec = (int(base_min) * 60) + (int(inc_sec) * 40)
        return tempo_totale_sec
    except Exception as e:
        return np.nan

# 2. Caricamento dei due dataset separati dal Notebook 01
# Questo garantisce che il Test Set subisca le stesse trasformazioni ma non si mescoli
df_train = pd.read_csv('train_cleaned.csv')
df_test = pd.read_csv('test_cleaned.csv')

# 3. Calcolo del tempo stimato
df_train['tempo_stimato_sec'] = df_train['increment_code'].apply(stima_tempo_lichess)
df_test['tempo_stimato_sec'] = df_test['increment_code'].apply(stima_tempo_lichess)

#Calcolo della lunghezza reale della partita
#Ci serve contare gli spazi nella stringa 'moves' per avere il numero di semimosse
df_train['numero_mosse_reali'] = df_train['moves'].apply(lambda x: len(str(x).split()))
df_test['numero_mosse_reali'] = df_test['moves'].apply(lambda x: len(str(x).split()))

print(f"Train Set preparato: {df_train.shape[0]} partite.")
print(f"Test Set preparato: {df_test.shape[0]} partite.")
print("Feature 'tempo_stimato_sec' e 'numero_mosse_reali' calcolate con successo.")

1. CARICAMENTO E PREPARAZIONE DATI
Train Set preparato: 14580 partite.
Test Set preparato: 3645 partite.
Feature 'tempo_stimato_sec' e 'numero_mosse_reali' calcolate con successo.


Calcolo delle varie features, viene tracciato se è stato fatto l'arrocco dalle moves, tramite O-O

In [ ]:
# Installa la libreria se non è già presente
!pip install python-chess
import chess
import pandas as pd

print("2. INIZIO ESTRAZIONE POSIZIONALE (FEATURE ENGINEERING) ")

# 1. MOTORE DI ESTRAZIONE POSIZIONALE
def estrai_features_posizionali(moves_string, n_semimosse):
    board = chess.Board()
    mosse = str(moves_string).split()

    # Variabili per tracciare lo stato dell'arrocco durante la simulazione
    ha_arroccato_b = 0
    ha_arroccato_n = 0

    for i in range(min(n_semimosse, len(mosse))):
        mossa_san = mosse[i]
        # Intercetta le mosse di arrocco (inclusi casi con scacco + o matto #)
        if "O-O" in mossa_san:
            if board.turn == chess.WHITE:
                ha_arroccato_b = 1
            else:
                ha_arroccato_n = 1

        board.push_san(mossa_san)

    # Vantaggio Materiale
    valori = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9}
    mat_b = sum(valori.get(p.piece_type, 0) for p in board.piece_map().values() if p.color == chess.WHITE)
    mat_n = sum(valori.get(p.piece_type, 0) for p in board.piece_map().values() if p.color == chess.BLACK)
    vantaggio_materiale = mat_b - mat_n

    # Mobilità Pseudo-Legale
    board_opp = board.copy()
    board_opp.turn = not board.turn

    if board.turn == chess.WHITE:
        mob_b = board.pseudo_legal_moves.count()
        mob_n = board_opp.pseudo_legal_moves.count()
    else:
        mob_b = board_opp.pseudo_legal_moves.count()
        mob_n = board.pseudo_legal_moves.count()

    vantaggio_mobilita = mob_b - mob_n

    # Diritti di Arrocco Residui
    arr_b = int(board.has_castling_rights(chess.WHITE))
    arr_n = int(board.has_castling_rights(chess.BLACK))

    # Scacco (Attivo solo per il Re del lato al tratto)
    scacco_b = 1 if board.is_check() and board.turn == chess.WHITE else 0

    # Sviluppo Pezzi Leggeri
    sviluppo_b = sum(1 for sq in board.pieces(chess.KNIGHT, chess.WHITE) if sq not in [chess.B1, chess.G1]) + \
                 sum(1 for sq in board.pieces(chess.BISHOP, chess.WHITE) if sq not in [chess.C1, chess.F1])
    sviluppo_n = sum(1 for sq in board.pieces(chess.KNIGHT, chess.BLACK) if sq not in [chess.B8, chess.G8]) + \
                 sum(1 for sq in board.pieces(chess.BISHOP, chess.BLACK) if sq not in [chess.C8, chess.F8])
    vantaggio_sviluppo = sviluppo_b - sviluppo_n

    return vantaggio_materiale, vantaggio_mobilita, arr_b, arr_n, ha_arroccato_b, ha_arroccato_n, scacco_b, vantaggio_sviluppo


# 2. ESECUZIONE, FILTRO E PULIZIA DEL DATA LEAKAGE
soglie = [10, 20, 30, 40]
datasets = {'train': df_train, 'test': df_test}

# Rimuovo il leakage, gli ID e le colonne testuali (aperture) per avere file CSV 100% numerici.
# Le aperture potranno essere recuperate in futuro tramite l'ID della partita dal file originale.
colonne_da_rimuovere = ['moves', 'numero_mosse_reali', 'opening_ply', 'increment_code', 'white_id', 'black_id', 'opening_eco', 'opening_name']

for split_name, df_current in datasets.items():
    print(f"\n--- ELABORAZIONE DEL {split_name.upper()} SET ---")

    for N in soglie:
        # Filtro dinamico
        df_N = df_current[df_current['numero_mosse_reali'] > N].copy()
        print(f"Estrazione N={N}: restano {df_N.shape[0]} partite valide.")

        # Estrazione feature
        risultati = df_N['moves'].apply(lambda x: estrai_features_posizionali(x, N))

        # Associazione nuove colonne
        col_names = [f'mat_{N}', f'mob_{N}', f'arr_b_{N}', f'arr_n_{N}', f'ha_arroccato_b_{N}', f'ha_arroccato_n_{N}', f'scacco_b_{N}', f'sviluppo_{N}']
        df_N[col_names] = pd.DataFrame(risultati.tolist(), index=df_N.index)

        # Pulizia Data Leakage e Colonne Testuali
        df_N = df_N.drop(columns=[col for col in colonne_da_rimuovere if col in df_N.columns], errors='ignore')

        # Salvataggio
        nome_file = f"{split_name}_features_N{N}.csv"
        df_N.to_csv(nome_file, index=False)

print("\nOPERAZIONE COMPLETATA: FILE CSV GENERATI")

2. INIZIO ESTRAZIONE POSIZIONALE (FEATURE ENGINEERING) 

--- ELABORAZIONE DEL TRAIN SET ---
Estrazione N=10: restano 14107 partite valide.
Estrazione N=20: restano 13279 partite valide.
Estrazione N=30: restano 11967 partite valide.
Estrazione N=40: restano 10135 partite valide.

--- ELABORAZIONE DEL TEST SET ---
Estrazione N=10: restano 3521 partite valide.
Estrazione N=20: restano 3340 partite valide.
Estrazione N=30: restano 3018 partite valide.
Estrazione N=40: restano 2617 partite valide.

OPERAZIONE COMPLETATA: FILE CSV GENERATI


Feature aggiunte = tempo stimato in secondi,
mat_N: la differenza tra punteggio materiale bianco - materiale nero;
mobN= La differenza tra le mosse pseudolegali del bianco e del nero;
arr_b_N: Variabile binaria (1 o 0) che indica se il Bianco ha ancora i diritti legali residui per arrocchare nel futuro.
arr_n_N:Variabile binaria (1 o 0) che indica se il Nero ha ancora i diritti legali residui per arrocchare nel futuro.
ha_arroccato_b_N:La nuova variabile binaria (1 o 0) che certifica se il Bianco ha già eseguito la mossa di arrocco entro la soglia N.
ha_arroccato_n_N:La nuova variabile binaria (1 o 0) che certifica se il Bianco ha già eseguito la mossa di arrocco entro la soglia N.
scacco_b_N:Variabile binaria (1 o 0) che segnala se il Re Bianco si trova fisicamente sotto attacco all'istante N.
sviluppo_N: l vantaggio di sviluppo (Bianco - Nero) calcolato contando Cavalli e Alfieri che hanno abbandonato la casa di partenza.

